In [52]:
import numpy as np
import pandas as pd
from itertools import combinations_with_replacement


In [67]:
def load_damped_oscillator_csv(filepath):
    df = pd.read_csv(filepath)

    
    t = df["t"].values
    X = df[["x", "v"]].values
    X_dot = df[["x_dot", "v_dot"]].values

    return t, X, X_dot




In [54]:
def estimate_derivatives_central_difference(X, t):
    
    X_dot_est = np.zeros_like(X)
    dt = t[1] - t[0]

    X_dot_est[1:-1] = (X[2:] - X[:-2]) / (2 * dt)
    X_dot_est[0] = (X[1] - X[0]) / dt
    X_dot_est[-1] = (X[-1] - X[-2]) / dt

    return X_dot_est




In [55]:
def build_candidate_library(
    X,
    poly_degree=1,
    variable_names=None,
    include_constant=True,
    include_sine=False,
    include_cosine=False,
    include_exp=False
):
    
    m, n = X.shape

    if variable_names is None:
        variable_names = [f"x{i+1}" for i in range(n)]

    Theta_columns = []
    feature_names = []

    if include_constant:
        Theta_columns.append(np.ones(m))
        feature_names.append("1")

    for d in range(1, poly_degree + 1):
        for combo in combinations_with_replacement(range(n), d):
            term = np.ones(m)
            power_count = {}

            for idx in combo:
                term *= X[:, idx]
                power_count[idx] = power_count.get(idx, 0) + 1

            name_parts = []
            for idx in sorted(power_count.keys()):
                power = power_count[idx]
                var = variable_names[idx]
                if power == 1:
                    name_parts.append(var)
                else:
                    name_parts.append(f"{var}^{power}")

            Theta_columns.append(term)
            feature_names.append(" ".join(name_parts))

    for i in range(n):
        var = variable_names[i]
        if include_sine:
            Theta_columns.append(np.sin(X[:, i]))
            feature_names.append(f"sin({var})")
        if include_cosine:
            Theta_columns.append(np.cos(X[:, i]))
            feature_names.append(f"cos({var})")

    for i in range(n):
        var = variable_names[i]
        if include_exp:
            Theta_columns.append(np.exp(X[:, i]))
            feature_names.append(f"exp({var})")

    Theta = np.column_stack(Theta_columns)
    return Theta, feature_names




In [56]:
def sindy_stlsq(Theta, X_dot, threshold=0.05, max_iter=10):
    
    Xi = np.linalg.lstsq(Theta, X_dot, rcond=None)[0]

    for _ in range(max_iter):
        Xi_old = Xi.copy()

        Xi[np.abs(Xi) < threshold] = 0.0

        for j in range(X_dot.shape[1]):
            big_idx = np.abs(Xi[:, j]) >= threshold

            if np.sum(big_idx) == 0:
                continue

            Xi[big_idx, j] = np.linalg.lstsq(
                Theta[:, big_idx],
                X_dot[:, j],
                rcond=None
            )[0]

        old_support = np.abs(Xi_old) >= threshold
        new_support = np.abs(Xi) >= threshold
        if np.array_equal(old_support, new_support):
            break

    return Xi




In [57]:
def equation_string(Xi, feature_names, eq_index, state_name):
    
    terms = []
    for i, coef in enumerate(Xi[:, eq_index]):
        if abs(coef) > 1e-12:
            terms.append(f"{coef:.6f}*{feature_names[i]}")
    rhs = " + ".join(terms) if terms else "0"
    return f"{state_name} = {rhs}"




In [58]:
def print_discovered_equations(Xi, feature_names, state_names=None):
    if state_names is None:
        state_names = [f"x{i+1}_dot" for i in range(Xi.shape[1])]

    for j in range(Xi.shape[1]):
        print(equation_string(Xi, feature_names, j, state_names[j]))




In [59]:
def get_active_terms(Xi, feature_names, eq_index):
    
    active = set()
    for i, coef in enumerate(Xi[:, eq_index]):
        if abs(coef) > 1e-12:
            active.add(feature_names[i])
    return active




In [60]:
def evaluate_recovery(Xi, feature_names):
    
    true_support_xdot = {"v"}
    true_support_vdot = {"x", "v"}

    recovered_xdot = get_active_terms(Xi, feature_names, eq_index=0)
    recovered_vdot = get_active_terms(Xi, feature_names, eq_index=1)

    xdot_correct = recovered_xdot == true_support_xdot
    vdot_correct = recovered_vdot == true_support_vdot
    full_correct = xdot_correct and vdot_correct

    return {
        "active_terms_x_dot": len(recovered_xdot),
        "active_terms_v_dot": len(recovered_vdot),
        "x_dot_support_correct": xdot_correct,
        "v_dot_support_correct": vdot_correct,
        "full_system_correct": full_correct,
    }




In [61]:
def write_results_file(results_rows, equation_blocks, output_file="result.txt"):
    
    df = pd.DataFrame(results_rows)

    with open(output_file, "w", encoding="utf-8") as f:
        f.write("SINDy Results Summary\n")
        f.write("=" * 80 + "\n\n")

        f.write("Summary Table\n")
        f.write("-" * 80 + "\n")
        f.write(df.to_string(index=False))
        f.write("\n\n")

        f.write("Recovered Equations\n")
        f.write("-" * 80 + "\n\n")

        for block in equation_blocks:
            f.write(f"Noise level: {block['noise_level']}\n")
            f.write(f"Library: {block['library_name']}\n")
            f.write(f"{block['x_dot_eq']}\n")
            f.write(f"{block['v_dot_eq']}\n")
            f.write("\n" + "-" * 80 + "\n\n")

    print("Results written to " + str(output_file))



In [62]:
def build_true_coefficient_matrix(feature_names):
    
    Xi_true = np.zeros((len(feature_names), 2))

    for i, name in enumerate(feature_names):
        if name == "v":
            Xi_true[i, 0] = 1.0

        if name == "x":
            Xi_true[i, 1] = -2.0
        elif name == "v":
            Xi_true[i, 1] = -0.2

    return Xi_true




In [63]:
def relative_l2_error(true_vec, recovered_vec, eps=1e-12):
    
    return np.linalg.norm(true_vec - recovered_vec) / (np.linalg.norm(true_vec) + eps)




In [64]:
def relative_frobenius_error(Xi_true, Xi_recovered, eps=1e-12):
    
    return np.linalg.norm(Xi_true - Xi_recovered, ord='fro') / (np.linalg.norm(Xi_true, ord='fro') + eps)




In [66]:
def evaluate_coefficient_error(Xi, feature_names):
    Xi_true = build_true_coefficient_matrix(feature_names)

    coef_error_x_dot = relative_l2_error(Xi_true[:, 0], Xi[:, 0])
    coef_error_v_dot = relative_l2_error(Xi_true[:, 1], Xi[:, 1])
    coef_error_total = relative_frobenius_error(Xi_true, Xi)

    return {
        "coef_error_x_dot": coef_error_x_dot,
        "coef_error_v_dot": coef_error_v_dot,
        "coef_error_total": coef_error_total,
    }

threshold = 0.05

datasets = {
    0.00: "damped_oscillator_data.csv",
    0.01: "damped_oscillator_data_0.01.csv",
    0.05: "damped_oscillator_data_0.05.csv",
    0.10: "damped_oscillator_data_0.1.csv",
}

library_configs = [
    {
        "name": "Polynomial degree 3",
        "poly_degree": 3,
        "include_sine": False,
        "include_cosine": False,
        "include_exp": False,
    },
    {
        "name": "Polynomial degree 3 + trig",
        "poly_degree": 3,
        "include_sine": True,
        "include_cosine": True,
        "include_exp": False,
    },
    {
        "name": "Polynomial degree 3 + exp",
        "poly_degree": 3,
        "include_sine": False,
        "include_cosine": False,
        "include_exp": True,
    },
    {
        "name": "Polynomial degree 3 + trig + exp",
        "poly_degree": 3,
        "include_sine": True,
        "include_cosine": True,
        "include_exp": True,
    },
]

results_rows = []
equation_blocks = []

for noise_std, filepath in datasets.items():
    print("\n" + "#" * 80)
    print("Noise level = " + str(noise_std))
    print("#" * 80)

    t, X, X_dot_true = load_damped_oscillator_csv(filepath)
    X_dot_est = estimate_derivatives_central_difference(X, t)

    for config in library_configs:
        print("\n" + "=" * 60)
        print("Testing library: " + str(config['name']))

        Theta, feature_names = build_candidate_library(
            X,
            poly_degree=config["poly_degree"],
            variable_names=["x", "v"],
            include_constant=True,
            include_sine=config["include_sine"],
            include_cosine=config["include_cosine"],
            include_exp=config["include_exp"],
        )

        Xi = sindy_stlsq(Theta, X_dot_est, threshold=threshold, max_iter=10)

        print("Theta shape:", Theta.shape)
        print("Discovered equations:")
        print_discovered_equations(
            Xi,
            feature_names,
            state_names=["x_dot", "v_dot"]
        )

        x_dot_eq = equation_string(Xi, feature_names, 0, "x_dot")
        v_dot_eq = equation_string(Xi, feature_names, 1, "v_dot")

        summary = evaluate_recovery(Xi, feature_names)
        coef_errors = evaluate_coefficient_error(Xi, feature_names)

        results_rows.append({
            "noise_level": noise_std,
            "library": config["name"],
            "active_terms_x_dot": summary["active_terms_x_dot"],
            "active_terms_v_dot": summary["active_terms_v_dot"],
            "x_dot_support_correct": summary["x_dot_support_correct"],
            "v_dot_support_correct": summary["v_dot_support_correct"],
            "full_system_correct": summary["full_system_correct"],
            "coef_error_x_dot": coef_errors["coef_error_x_dot"],
            "coef_error_v_dot": coef_errors["coef_error_v_dot"],
            "coef_error_total": coef_errors["coef_error_total"],
        })

        equation_blocks.append({
            "noise_level": noise_std,
            "library_name": config["name"],
            "x_dot_eq": x_dot_eq,
            "v_dot_eq": v_dot_eq,
        })

write_results_file(results_rows, equation_blocks, output_file="result.txt")



################################################################################
Noise level = 0.0
################################################################################

Testing library: Polynomial degree 3
Theta shape: (2501, 10)
Discovered equations:
x_dot = 0.999967*v
v_dot = -1.999927*x + -0.199986*v

Testing library: Polynomial degree 3 + trig
Theta shape: (2501, 14)
Discovered equations:
x_dot = 0.999967*v
v_dot = -1.999927*x + -0.199986*v

Testing library: Polynomial degree 3 + exp
Theta shape: (2501, 12)
Discovered equations:
x_dot = 0.999967*v
v_dot = -1.999927*x + -0.199986*v

Testing library: Polynomial degree 3 + trig + exp
Theta shape: (2501, 16)
Discovered equations:
x_dot = 0.999967*v
v_dot = -1.999927*x + -0.199986*v

################################################################################
Noise level = 0.01
################################################################################

Testing library: Polynomial degree 3
Theta shape: (2501, 10)
D